# Discovery + Silver: `crm.opportunities`

Pendiente marcada en `docs/calidad_datos.md`: `close_date < created_at` en ~34% de las filas. Antes de decidir que hacer, medimos si esto se concentra en algun `stage` particular (mismo criterio que en `courses` y `subscriptions`: medir antes de asumir).

In [1]:
import sys
from pathlib import Path
sys.path.append("/home/jovyan/work/src")

import pandas as pd
from utils.db import get_engine, get_psycopg2_connection

engine = get_engine()
SQL_SILVER = Path("/home/jovyan/work/sql/silver")

def run_sql_file(path):
    sql = path.read_text()
    conn = get_psycopg2_connection()
    try:
        with conn.cursor() as cur:
            cur.execute(sql)
        conn.commit()
        print(f"OK: {path.name} ejecutado")
    finally:
        conn.close()

df = pd.read_sql("SELECT * FROM bronze.crm__opportunities", engine)
df.shape

(3000, 10)

## 1. Forma general

In [2]:
print(df.dtypes)
df.head()

opportunity_id            object
name                      object
stage                     object
amount                    object
close_date                object
created_at                object
account_id                object
_source_file              object
_ingested_at      datetime64[ns]
_dag_run_id               object
dtype: object


,opportunity_id,name,stage,amount,close_date,created_at,account_id,_source_file,_ingested_at,_dag_run_id
0,OPP-0000001,Deal 0000001,won,5746.66,2023-11-30,2022-03-19 16:45:26,ACC-0002022,crm/opportunities.csv,2026-07-21 09:50:02.443080,manual__2026-07-21T09:49:58+00:00
1,OPP-0000002,Deal 0000002,negotiation,24990.51,2023-09-28,2025-05-25 03:05:15,ACC-0000582,crm/opportunities.csv,2026-07-21 09:50:02.443080,manual__2026-07-21T09:49:58+00:00
2,OPP-0000003,Deal 0000003,qualification,11892.82,2024-07-02,2023-12-23 16:49:50,ACC-0002591,crm/opportunities.csv,2026-07-21 09:50:02.443080,manual__2026-07-21T09:49:58+00:00
3,OPP-0000004,Deal 0000004,proposal,31211.68,2023-09-19,2023-05-19 04:41:38,ACC-0004529,crm/opportunities.csv,2026-07-21 09:50:02.443080,manual__2026-07-21T09:49:58+00:00
4,OPP-0000005,Deal 0000005,proposal,27434.07,2023-08-30,2024-11-30 06:25:22,ACC-0003484,crm/opportunities.csv,2026-07-21 09:50:02.443080,manual__2026-07-21T09:49:58+00:00


## 2. Nulos, duplicados e integridad referencial

In [3]:
print("Nulos por columna:")
print(df.isna().sum())
print()
print("opportunity_id duplicados:", df["opportunity_id"].duplicated().sum())

accounts = pd.read_sql("SELECT account_id FROM silver.crm__accounts", engine)
print("account_id huerfanos:", (~df["account_id"].isin(accounts["account_id"])).sum())

Nulos por columna:
opportunity_id    0
name              0
stage             0
amount            0
close_date        0
created_at        0
account_id        0
_source_file      0
_ingested_at      0
_dag_run_id       0
dtype: int64

opportunity_id duplicados: 0
account_id huerfanos: 0


## 3. `close_date < created_at`: medirlo por stage

In [4]:
created = pd.to_datetime(df["created_at"])
close = pd.to_datetime(df["close_date"])
antes = close < created

print("stage:")
print(df["stage"].value_counts())
print()
print("close_date < created_at (total):", antes.sum(), f"({antes.mean()*100:.1f}%)")
print()
print("% de close_date < created_at, por stage:")
print(df.assign(antes=antes).groupby("stage")["antes"].mean() * 100)

stage:
stage
prospect         621
qualification    611
proposal         569
won              476
negotiation      420
lost             303
Name: count, dtype: int64

close_date < created_at (total): 1029 (34.3%)

% de close_date < created_at, por stage:
stage
lost             31.353135
negotiation      31.904762
proposal         36.203866
prospect         34.943639
qualification    33.060556
won              36.764706
Name: antes, dtype: float64


## 4. `amount`: rango de valores

In [5]:
amount = pd.to_numeric(df["amount"], errors="coerce")
print("amount <= 0:", (amount <= 0).sum())
print(amount.describe())

amount <= 0: 0
count      3000.000000
mean      37921.932363
std       47713.624101
min         699.920000
25%       11890.805000
50%       23151.380000
75%       45815.445000
max      687007.800000
Name: amount, dtype: float64


## 5. Conclusion

El porcentaje de `close_date < created_at` es similar entre **todos** los stages (abiertos y cerrados por igual) -- no se concentra en `won`/`lost` (donde tendria mas sentido que fuera un error real de cierre) ni en los stages abiertos (donde seria solo una fecha objetivo optimista). Al ser uniforme, tratamos `close_date` como **fecha objetivo/estimada**, no como fecha real de cierre -- consistente con el criterio ya documentado en `docs/decisiones.md`.

Regla: **no se anula ni se descarta ninguna fila**, pero se agrega un flag `_close_date_before_created` para que quede trazable en silver (por si en el analisis de negocio se quiere filtrar o ponderar esas oportunidades distinto).

## 6. Limpieza con pandas

In [6]:
df_silver = df[["opportunity_id", "account_id", "name", "stage", "amount", "created_at", "close_date"]].copy()

df_silver["name"] = df_silver["name"].str.strip()
df_silver["stage"] = df_silver["stage"].str.strip().str.lower()
df_silver["amount"] = pd.to_numeric(df_silver["amount"], errors="raise")
df_silver["created_at"] = pd.to_datetime(df_silver["created_at"])
df_silver["close_date"] = pd.to_datetime(df_silver["close_date"]).dt.date

df_silver["_close_date_before_created"] = pd.to_datetime(df_silver["close_date"]) < df_silver["created_at"]

df_silver.head()

,opportunity_id,account_id,name,stage,amount,created_at,close_date,_close_date_before_created
0,OPP-0000001,ACC-0002022,Deal 0000001,won,5746.66,2022-03-19 16:45:26,2023-11-30,False
1,OPP-0000002,ACC-0000582,Deal 0000002,negotiation,24990.51,2025-05-25 03:05:15,2023-09-28,True
2,OPP-0000003,ACC-0002591,Deal 0000003,qualification,11892.82,2023-12-23 16:49:50,2024-07-02,False
3,OPP-0000004,ACC-0004529,Deal 0000004,proposal,31211.68,2023-05-19 04:41:38,2023-09-19,False
4,OPP-0000005,ACC-0003484,Deal 0000005,proposal,27434.07,2024-11-30 06:25:22,2023-08-30,True


## 7. Validar antes de escribir

In [7]:
assert len(df_silver) == len(df)
assert df_silver["opportunity_id"].is_unique
assert df_silver["account_id"].isin(accounts["account_id"]).all()
assert df_silver["amount"].gt(0).all()
print("OK:", len(df_silver), "filas listas para silver")

OK: 3000 filas listas para silver


## 8. Escribir en `silver.crm__opportunities`

In [8]:
df_silver["_silver_loaded_at"] = pd.Timestamp.utcnow()

run_sql_file(SQL_SILVER / "crm.sql")

conn = get_psycopg2_connection()
with conn.cursor() as cur:
    cur.execute("TRUNCATE TABLE silver.crm__opportunities CASCADE;")
conn.commit()
conn.close()

df_silver.to_sql(
    "crm__opportunities",
    engine,
    schema="silver",
    if_exists="append",
    index=False,
    method="multi",
    chunksize=2000,
)
print("Escrito en silver.crm__opportunities")

OK: crm.sql ejecutado


Escrito en silver.crm__opportunities


## 9. Verificar

In [9]:
check = pd.read_sql("SELECT * FROM silver.crm__opportunities LIMIT 5", engine)
print(pd.read_sql("SELECT count(*) AS filas, count(*) FILTER (WHERE _close_date_before_created) AS con_flag FROM silver.crm__opportunities", engine))
check

   filas  con_flag
0   3000      1029


,opportunity_id,account_id,name,stage,amount,created_at,close_date,_close_date_before_created,_silver_loaded_at
0,OPP-0000001,ACC-0002022,Deal 0000001,won,5746.66,2022-03-19 16:45:26,2023-11-30,False,2026-07-21 09:50:57.978534+00:00
1,OPP-0000002,ACC-0000582,Deal 0000002,negotiation,24990.51,2025-05-25 03:05:15,2023-09-28,True,2026-07-21 09:50:57.978534+00:00
2,OPP-0000003,ACC-0002591,Deal 0000003,qualification,11892.82,2023-12-23 16:49:50,2024-07-02,False,2026-07-21 09:50:57.978534+00:00
3,OPP-0000004,ACC-0004529,Deal 0000004,proposal,31211.68,2023-05-19 04:41:38,2023-09-19,False,2026-07-21 09:50:57.978534+00:00
4,OPP-0000005,ACC-0003484,Deal 0000005,proposal,27434.07,2024-11-30 06:25:22,2023-08-30,True,2026-07-21 09:50:57.978534+00:00
